Létrehozta: Panna

Adatok beolvasása:

In [ ]:
from pathlib import Path
from services.data_loader import load_all_messages

PROJECT_ROOT = Path(globals().get("project_root", ".")).resolve()
JSON_DIR = PROJECT_ROOT / "data" / "json"

merged, names = load_all_messages(JSON_DIR)

1. Mi volt Anita utolsó üzenete? (1 pont)

In [ ]:
uzik = []
for i in merged:
    if i['sender_name'] == "Anita Kaderják":
        uzik.append(i['timestamp_ms'])
uzik.sort()
for i in merged:
    if i['sender_name'] == "Anita Kaderják":
        if i['timestamp_ms'] == uzik[-1]:
            print(i['content'])

2. Kiket távolított el eddig Attila a csoportból? (2 pont)

In [ ]:
kickelt = []

for uzenet in merged:
    if not isinstance(uzenet, dict):
        continue

    content = uzenet.get("content")
    sender = uzenet.get("sender_name")

    if isinstance(content, str) and sender == "Attila Chikan":
        if "eltávolította" in content:
            kickelt.append(uzenet)

3. Mi a legtöbbet használt reakció? (2 pont)

In [ ]:
jelek = {}

for i in merged:
    reactions = i.get('reactions', [])

    if not isinstance(reactions, list):
        continue

    for j in reactions:
        if not isinstance(j, dict):
            continue

        jel = j.get('reaction')
        if not jel:
            continue

        jelek[jel] = jelek.get(jel, 0) + 1

sorted_jelek = sorted(jelek.items(), key=lambda x: x[1], reverse=True)

for reaction, count in sorted_jelek[:5]:
    print(reaction, count)

4. Összesen melyik évfolyam mennyi üzenetet küldött? (4 pont)

In [ ]:
evfolyamok = {}

for i in merged:
    name = i['sender_name']

    for j in names:
        if j['name'] == name:
            osztaly = j['class']
            evfolyamok[osztaly] = evfolyamok.get(osztaly, 0) + 1

sorted_evfolyamok = sorted(
    evfolyamok.items(),
    key=lambda x: x[1],
    reverse=True
)

print(sorted_evfolyamok)

5. Ki írta a legtöbb üzenetet? (2 pont)

In [ ]:
ossz_uz = {}

for i in merged:
    if not isinstance(i, dict):
        continue

    name = i.get('sender_name')
    if not name:
        continue

    ossz_uz[name] = ossz_uz.get(name, 0) + 1

sorted_ossz_uz = sorted(
    ossz_uz.items(),
    key=lambda x: x[1],
    reverse=True
)

for name, count in sorted_ossz_uz[:5]:
    print(name, count)

6. Ki vonta vissza a legtöbb üzenetet? (1 pont)

In [ ]:
visszavonok = {}

for i in merged:
    if not isinstance(i, dict):
        continue

    if i.get('is_unsent', 0) > 0:
        name = i.get('sender_name')
        if name:
            visszavonok[name] = visszavonok.get(name, 0) + 1

sorted_visszavonok = sorted(
    visszavonok.items(),
    key=lambda x: x[1],
    reverse=True
)

for name, count in sorted_visszavonok[:5]:
    print(name, count)

7. Mennyi a legtöbb idő ami eltelt két egymást követő üzenet között? (4 pont)

In [ ]:
timestamps = sorted(i['timestamp_ms'] for i in merged)

szunetek = [
    timestamps[i+1] - timestamps[i]
    for i in range(len(timestamps) - 1)
]

top5 = sorted(szunetek, reverse=True)[:5]

for x in top5:
    print(x)

8. Ki írja átlagosan a leghosszabb üzeneteket? Itt nem számítanak a 0 karakter hosszú üzenetek. (2 pont)

In [ ]:
hosszu = {}

for i in merged:
    name = i['sender_name']
    content = i.get('content', "")

    if content:
        hosszu.setdefault(name, []).append(len(content))

atlagok = {
    name: sum(lengths) / len(lengths)
    for name, lengths in hosszu.items()
    if lengths
}

top5 = sorted(atlagok.items(), key=lambda x: x[1], reverse=True)[:5]

for name, avg in top5:
    print(name, avg)

9. Ki lépett ki a legtöbbször a jeszk momentsből (2 pont)

In [ ]:
ki = {}

for i in merged:
    content = i.get('content')

    if content and 'kilépett' in content:
        name = i['sender_name']
        ki[name] = ki.get(name, 0) + 1

top5 = sorted(
    ki.items(),
    key=lambda x: (-x[1], x[0])
)[:5]

for name, count in top5:
    print(name, count)

10. Ki küldte legtöbb képet a jeszk momentsbe? (2 pont)

In [ ]:
kepkuldok = {}

for i in merged:
    photos = i.get('photos', 0)

    # FIX: normalize type
    if isinstance(photos, list):
        photos = len(photos)
    elif not isinstance(photos, int):
        photos = 0

    if photos > 0:
        name = i.get('sender_name', 'unknown')
        kepkuldok[name] = kepkuldok.get(name, 0) + photos

top5 = sorted(kepkuldok.items(), key=lambda x: (-x[1], x[0]))[:5]

for name, count in top5:
    print(name, count)

11. Hányan vannak, akik pontosan n évben küldtek üzenetet ($ n = 1, ... 7 $) (2 pont, ábrázolásért +1)

In [ ]:
years = [merged]

user_years = {}

for year_index, data in enumerate(years, start=2017):
    for msg in data:
        name = msg['sender_name']

        if name not in user_years:
            user_years[name] = set()

        user_years[name].add(year_index)

year_counts = [len(s) for s in user_years.values()]

result = {i: 0 for i in range(1, 10)}

for c in year_counts:
    if 1 <= c <= 9:
        result[c] += 1

for i in range(1, 10):
    print(i, result[i])

12. Melyik szó volt a legnépszerűbb az egyes években? (4 pont)

In [ ]:
years = {}

for msg in merged:
    if not isinstance(msg, dict):
        continue

    year = msg.get("year")
    if year is None:
        continue

    years.setdefault(year, []).append(msg)

for year, data in years.items():
    freq = {}

    for msg in data:
        if not isinstance(msg, dict):
            continue

        text = msg.get("content_clean", "")

        if not isinstance(text, str):
            continue

        for word in text.lower().split():
            freq[word] = freq.get(word, 0) + 1

    top5 = sorted(freq.items(), key=lambda x: (-x[1], x[0]))[:5]

    print(year)
    for word, count in top5:
        print(word, count)
    print()

13. Mi volt az az üzenet, amelyik a legtöbb reakciót kapta? (2 pont)

In [ ]:
reakciok = []

for i in merged:
    szam = len(i.get('reactions', []))
    reakciok.append(szam)

reakciok.sort()
top5_vals = reakciok[-5:][::-1]  # biggest → smallest

# collect matching posts
top_dicts = {v: [] for v in top5_vals}

for i in merged:
    szam = len(i.get('reactions', []))
    if szam in top_dicts:
        top_dicts[szam].append(i)

# print results
for v in top5_vals:
    print("\nREACTIONS =", v)
    print(top_dicts[v])

14. Meddig tartott a leghosszabb hívás? (2 pont)

In [ ]:
timestamps = []

for i in merged:
    content = i.get("content", "")
    if not isinstance(content, str):
        continue

    if (
        "hívást indított." in content
        or "videochatet indított." in content
        or "hívás befejeződött." in content
        or "videohívás véget ért." in content
        or "videóhívás véget ért." in content
    ):
        ts = i.get("timestamp_ms")
        if isinstance(ts, (int, float)):
            timestamps.append(ts)

timestamps.sort()

hivas_hossza = []
for i in range(0, len(timestamps) - 1, 2):
    start = timestamps[i]
    end = timestamps[i + 1]
    if end > start:
        hivas_hossza.append(end - start)

hivas_hossza.sort()

for x in hivas_hossza[-5:][::-1]:
    print(x / 1000)

511886.361
208449.718
1224.454
548.823
320.96


15. Ki érte el a legtöbb reakciót átlagosan a jeszk momentsben? (4 pont)

In [ ]:
nepszeruek = {}

for i in merged:
    name = i['sender_name']
    szam = len(i.get('reactions', []))

    if name not in nepszeruek:
        nepszeruek[name] = []

    nepszeruek[name].append(szam)

# compute averages
atlagok = {
    name: sum(values) / len(values)
    for name, values in nepszeruek.items()
    if values
}

top5 = sorted(atlagok.items(), key=lambda x: (-x[1], x[0]))[:5]

for name, avg in top5:
    print(name, avg)

Zoltán Deskó 9.333333333333334
Marton Kovesdy 8.0
Andor Berta 7.0625
Bendeguz Varadi 7.0
János Varga 6.512345679012346


16. Melyik nap küldték a legtöbb üzenetet a jeszk momentsbe? (1 pont)

In [ ]:
from datetime import datetime

osszesites = {}

for i in merged:
    ts = i.get("timestamp_ms")
    if not isinstance(ts, (int, float)):
        continue

    dt = datetime.fromtimestamp(ts / 1000)
    datum = dt.date()

    osszesites[datum] = osszesites.get(datum, 0) + 1

total_messages = len(merged)
unique_days = len(osszesites)

avg = total_messages / unique_days if unique_days else 0

print(avg)
print(total_messages)
print(unique_days)

1 2017:10:17 264
2 2019:2:6 213
3 2020:3:18 208
4 2025:7:1 155
5 2023:11:9 131


17. Mennyi üzenetet küldenek átlagosan egy nap a jeszk momentsbe? (1 pont)

In [13]:
osszesites = {}

for i in merged:
    datum = (i['year'], i['month'], i['day'])

    osszesites[datum] = osszesites.get(datum, 0) + 1

total_messages = len(merged)
unique_days = len(osszesites)

avg = total_messages / unique_days

print(avg)
print(total_messages)
print(unique_days)

22.896140350877193
65254
2850


18. Hányszor végzett valamilyen aktivitást 2025-ben egy jelenlegi DB vagy SZMT tag (az igazgatókat nem beleértve) (1 pont) ??

In [ ]:
print("lol")